# SHAP Feature Importance Analysis

Feature importance for the elastic net (standardized coefficients) and the GBM (SHAP TreeExplainer).

Both models trained on 1991–2019; evaluated/explained on 2005–2024.

**Primary model:** Elastic net with Δ-targets (`src/models/train_ridge.py`)  
**Comparison model:** LightGBM multi-output (`src/models/train_gb.py`)

In [ ]:
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.models.train_ridge import (
    load_data, get_X, NUMERIC_FEATURES, VARIETIES, TARGETS, TRAIN_CUTOFF
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
COLORS = {'Cabernet Sauvignon': '#8B1A1A', 'Pinot Noir': '#C04000', 'Chardonnay': '#D4AF37'}
print('Setup complete.')

In [ ]:
df = load_data()
train_df = df[df['year'] <= TRAIN_CUTOFF]
print(f'Training set: {train_df["year"].min()}–{TRAIN_CUTOFF} | {len(train_df)} rows')
print(f'Features: {NUMERIC_FEATURES}')

## Part 1 — Elastic Net: Standardized Coefficients

For a linear model, standardized coefficients (coef × feature std) directly measure feature importance: larger absolute value = stronger effect on the target delta.

Positive coefficient → feature pushes Δtarget up (warmer/drier year increases brix change).  
Negative coefficient → feature pushes Δtarget down.

In [ ]:
PARAMS = {
    'Cabernet Sauvignon': {'alpha': 1.0,  'l1_ratio': 0.1},
    'Pinot Noir':         {'alpha': 0.01, 'l1_ratio': 0.5},
    'Chardonnay':         {'alpha': 10.0, 'l1_ratio': 0.5},
}

FEATURE_LABELS = {
    'gdd':               'GDD (Apr–Oct)',
    'frost_days':        'Frost days',
    'heat_stress_days':  'Heat stress days',
    'tmax_veraison':     'Tmax veraison (Jul–Aug)',
    'precip_winter':     'Winter precip (mm)',
    'eto_season':        'Seasonal ETo (in)',
    'severity_score':    'Drought severity',
    'awc_r':             'Soil water capacity',
    'claytotal_r':       'Soil clay %',
    'brix_lag1':         'Brix (prior year)',
    'tons_crushed_lag1': 'Tons crushed (prior year)',
}

coef_data = {}

for variety in VARIETIES:
    sub = train_df[train_df['variety'] == variety].dropna(
        subset=NUMERIC_FEATURES + ['delta_brix', 'delta_tons_crushed']
    )
    X, scaler = get_X(sub, fit=True)
    coef_data[variety] = {}
    for tgt in TARGETS:
        m = ElasticNet(max_iter=5000, random_state=42, **PARAMS[variety])
        m.fit(X, sub[f'delta_{tgt}'].values)
        # Standardize by feature std so coefficients are comparable
        stds = np.array([sub[f].std() for f in NUMERIC_FEATURES])
        coef_data[variety][tgt] = m.coef_ * stds

print('Elastic net models fitted.')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharey=True)
labels = [FEATURE_LABELS[f] for f in NUMERIC_FEATURES]

for col, variety in enumerate(VARIETIES):
    for row, tgt in enumerate(TARGETS):
        ax = axes[row, col]
        coefs = coef_data[variety][tgt]
        colors = ['#c0392b' if c > 0 else '#2980b9' for c in coefs]
        ax.barh(labels, coefs, color=colors)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title(f'{variety}\nΔ{tgt.replace("_", " ").title()}', fontsize=10)
        ax.set_xlabel('Standardized coefficient')

fig.suptitle('Elastic Net — Standardized Feature Coefficients\n(red = positive effect on delta, blue = negative)', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'models' / 'coef_plot.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → models/coef_plot.png')

## Part 2 — GBM: SHAP TreeExplainer

Tree SHAP decomposes each prediction into additive feature contributions, capturing non-linear effects and interactions that linear coefficients cannot show.

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import OneHotEncoder
from src.models.train_gb import (
    load_data as gbm_load_data,
    build_feature_matrix as gbm_feature_matrix,
    NUMERIC_FEATURES as GBM_NUMERIC,
    CATEGORICAL_FEATURES, TRAIN_CUTOFF as GBM_CUTOFF
)

GBM_PARAMS = {
    'Cabernet Sauvignon': {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_samples': 3, 'subsample': 0.9},
    'Pinot Noir':         {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_samples': 5, 'subsample': 0.8},
    'Chardonnay':         {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 2, 'min_child_samples': 5, 'subsample': 1.0},
}

gbm_df = gbm_load_data()
gbm_models = {}
gbm_encoders = {}
gbm_X_trains = {}

for variety in VARIETIES:
    sub = gbm_df[(gbm_df['variety'] == variety) & (gbm_df['year'] <= GBM_CUTOFF)]
    sub = sub.dropna(subset=GBM_NUMERIC + TARGETS)
    X_train, enc = gbm_feature_matrix(sub, fit_encoder=True)
    y_train = sub[TARGETS].values
    model = MultiOutputRegressor(LGBMRegressor(verbose=-1, random_state=42, **GBM_PARAMS[variety]))
    model.fit(X_train, y_train)
    gbm_models[variety] = model
    gbm_encoders[variety] = enc
    gbm_X_trains[variety] = X_train

print('GBM models fitted.')

In [ ]:
# Compute SHAP values for brix estimator (index 0) per variety
shap_values_all = {}
shap_X_all = {}

for variety in VARIETIES:
    sub = gbm_df[(gbm_df['variety'] == variety) & (gbm_df['year'] >= 2005)].dropna(subset=GBM_NUMERIC)
    X_explain, _ = gbm_feature_matrix(sub, encoder=gbm_encoders[variety])
    brix_estimator = gbm_models[variety].estimators_[0]  # index 0 = brix
    explainer = shap.TreeExplainer(brix_estimator)
    sv = explainer.shap_values(X_explain)
    shap_values_all[variety] = sv
    shap_X_all[variety] = X_explain
    print(f'{variety}: SHAP computed on {len(X_explain)} years')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, variety in zip(axes, VARIETIES):
    plt.sca(ax)
    shap.summary_plot(
        shap_values_all[variety],
        shap_X_all[variety],
        feature_names=list(shap_X_all[variety].columns),
        plot_type='bar',
        max_display=10,
        show=False,
        color=COLORS[variety],
    )
    ax.set_title(variety)

fig.suptitle('GBM SHAP — Mean |SHAP| for Brix Prediction (2005–2024)', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'models' / 'shap_brix_bar.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → models/shap_brix_bar.png')

In [ ]:
# Waterfall plots for representative years: drought (2021), wet (2017), average (2012)
REPRESENTATIVE_YEARS = {'Drought (2021)': 2021, 'Wet (2017)': 2017, 'Average (2012)': 2012}
variety_wf = 'Cabernet Sauvignon'

sub = gbm_df[(gbm_df['variety'] == variety_wf) & (gbm_df['year'] >= 2005)].dropna(subset=GBM_NUMERIC).reset_index(drop=True)
X_explain, _ = gbm_feature_matrix(sub, encoder=gbm_encoders[variety_wf])
brix_estimator = gbm_models[variety_wf].estimators_[0]
explainer = shap.TreeExplainer(brix_estimator)
explanation = explainer(X_explain)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (label, year) in zip(axes, REPRESENTATIVE_YEARS.items()):
    idx = sub[sub['year'] == year].index
    if len(idx) == 0:
        ax.set_title(f'{label} — no data')
        continue
    plt.sca(ax)
    shap.waterfall_plot(explanation[idx[0]], max_display=8, show=False)
    ax.set_title(f'{label}\n({variety_wf})')

fig.suptitle('GBM SHAP Waterfall — Brix Prediction by Year Type', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'models' / 'shap_waterfall.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → models/shap_waterfall.png')

In [ ]:
# Variety comparison: top feature by mean |SHAP| per variety
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, variety in zip(axes, VARIETIES):
    sv = np.abs(shap_values_all[variety])
    mean_abs = sv.mean(axis=0)
    feat_names = list(shap_X_all[variety].columns)
    top_idx = np.argsort(mean_abs)[-8:]
    ax.barh([feat_names[i] for i in top_idx], mean_abs[top_idx], color=COLORS[variety])
    ax.set_title(variety)
    ax.set_xlabel('Mean |SHAP|')

fig.suptitle('Top 8 Features by Mean |SHAP| — Brix Prediction', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'models' / 'shap_variety_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Save SHAP values
out = ROOT / 'models' / 'shap_values.pkl'
with open(out, 'wb') as f:
    pickle.dump({'brix': shap_values_all, 'feature_names': {v: list(shap_X_all[v].columns) for v in VARIETIES}}, f)
print(f'Saved → {out.relative_to(ROOT)}')